# Module 8 · — Monitoring & Feedback: Drift Detection & Automated Alerting

**Architectural Layer:** Monitoring & Feedback  
**Toolchain:** EvidentlyAI · Prometheus · Grafana · scikit-learn  
**Objective:** Detect distributional drift, compute embedding-space degradation, and trigger automated retraining callbacks.

---

## 🧠 What is Model Monitoring and Why Is It Critical?

### The Fundamental Problem

**ML models degrade over time.** Unlike traditional software that works the same forever unless you change the code, ML models depend on DATA — and data changes.

**Real-world scenario:**  
A loan approval model was trained on pre-COVID data (2019). During COVID (2020), people's income patterns, spending habits, and employment status changed dramatically. The model kept approving loans based on outdated patterns, leading to massive default rates. Nobody noticed for MONTHS because nobody was monitoring.

### 🤔 Why Can't We Just Check Accuracy?

Because in production, you often **don't have ground truth labels** right away.

| Scenario | When You Get Labels | Monitoring Gap |
|----------|--------------------|---------|
| Loan default prediction | 12-36 months later | Model could be wrong for a year before you know |
| Ad click prediction | Minutes-hours | Short gap, but still delayed |
| Medical diagnosis | Weeks-months after follow-up | Critical — wrong diagnoses happen silently |
| Content recommendation | Never (no explicit feedback) | Must rely on proxy metrics |

**Solution: Monitor the INPUTS (data drift) as a proxy for output degradation.** If the input data changes significantly from what the model was trained on, the model's predictions are likely unreliable — even before you get labels.

### Types of Drift

| Drift Type | What Changes | Example | Detection Method |
|-----------|-------------|---------|------------------|
| **Data drift** | Input feature distributions | User age shifts from 35→28 average | KS test, PSI |
| **Concept drift** | Relationship between inputs and outputs | What predicts "good customer" changes | Monitor prediction accuracy |
| **Prediction drift** | Model output distribution | Approval rate shifts from 30%→50% | Compare prediction histograms |
| **Label drift** | Ground truth distribution | Actual default rate changes | Monitor when labels arrive |

### ⚖️ Trade-offs: How Often to Monitor?

| Frequency | Pros | Cons | Best For |
|-----------|------|------|----------|
| Real-time (every request) | Catch drift instantly | Very expensive compute | Safety-critical (medical, autonomous vehicles) |
| Hourly batches | Good balance | 1-hour detection delay | Most production systems |
| Daily batches | Low cost | Drift can go unnoticed for a day | Low-risk systems |
| Weekly | Minimal cost | Week-long blind spot | Research/experimental systems |

---

## ✅ Knowledge Check
### Q1: How does embedding drift differ from tabular feature drift?
<details><summary>Answer</summary>
Tabular features can be tested individually using standard statistical tests, while embeddings require checking multi-dimensional distribution shifts like centroid distance or MMD.
</details>
### Q2: Why measure input drift instead of waiting for label-based accuracy drops?
<details><summary>Answer</summary>
Label feedback in production often has a significant delay (e.g., loan defaults). Input drift catches potentially bad predictions immediately.
</details>


## 🔨 Practical Practice
### Exercise 1: Drift Calculation
Compute a Kolmogorov-Smirnov test manually on two synthetic numerical distributions to detect feature drift.
### Exercise 2: Shadow Deployment
Simulate a shadow deployment architecture that passes traffic to a new candidate model without affecting the user response payload.


## 📑 Table of Contents

* [🧠 What is Model Monitoring and Why Is It Critical?](#what-is-model-monitoring-and-why-is-it-critical)
  * [The Fundamental Problem](#the-fundamental-problem)
  * [🤔 Why Can't We Just Check Accuracy?](#why-cant-we-just-check-accuracy)
  * [Types of Drift](#types-of-drift)
  * [⚖️ Trade-offs: How Often to Monitor?](#trade-offs-how-often-to-monitor)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Traffic Simulation: Reference vs. Production](#1-traffic-simulation-reference-vs-production)
  * [🤔 What is "Reference" vs "Production" Data?](#what-is-reference-vs-production-data)
* [2 · Statistical Drift Detection](#2-statistical-drift-detection)
  * [🤔 How Do We Mathematically Detect Drift?](#how-do-we-mathematically-detect-drift)
  * [⚖️ Sensitivity Trade-off](#sensitivity-trade-off)
* [3 · Embedding-Space Drift Analysis](#3-embedding-space-drift-analysis)
  * [🤔 Why Monitor Embeddings Separately?](#why-monitor-embeddings-separately)
  * [Metrics for Embedding Drift](#metrics-for-embedding-drift)
* [4 · Automated Retraining Callback](#4-automated-retraining-callback)
  * [🤔 What Happens When Drift Is Detected?](#what-happens-when-drift-is-detected)
  * [🤔 What if We Auto-Retrain Too Often?](#what-if-we-auto-retrain-too-often)
  * [🤔 What if We Never Retrain?](#what-if-we-never-retrain)
* [5 -- A/B Testing & Experimentation for AI Models](#5-ab-testing-experimentation-for-ai-models)
  * [Why A/B Testing Is Different for AI](#why-ab-testing-is-different-for-ai)
  * [Deployment Strategies](#deployment-strategies)
  * [Shadow Mode: Running Models Without User Impact](#shadow-mode-running-models-without-user-impact)
* [6 -- Observability: OpenTelemetry & Structured Logging](#6-observability-opentelemetry-structured-logging)
  * [Why Standard Logging Is Not Enough for AI](#why-standard-logging-is-not-enough-for-ai)
  * [The Three Pillars of Observability](#the-three-pillars-of-observability)
  * [Structured Logging for AI Services](#structured-logging-for-ai-services)
  * [Key Metrics to Dashboard for AI Systems](#key-metrics-to-dashboard-for-ai-systems)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [🧠 Key Questions](#key-questions)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Machine Learning models degrade silently. A software bug throws a traceback; an ML data drift just casually lowers revenue by 12%. Seniors monitor input distributions (data drift) and output patterns continuously.

**Mental Model:** Software monitoring is a heart rate monitor (is the server alive?). ML monitoring is taking a blood test (are the internal properties of the data healthy and matching the training data?).

**Common Junior Pitfall:** Only monitoring latency and CPU. If the world changes (e.g., COVID hits and user buying behavior changes totally), the model latency is fine, but the predictions are 100% wrong.

---


In [ ]:
# Cell 1 — Install dependencies
!pip install -q evidently pandas numpy scikit-learn matplotlib scipy requests flask

## 1 · Traffic Simulation: Reference vs. Production

### 🤔 What is "Reference" vs "Production" Data?

- **Reference data** = the data your model was TRAINED on (the "normal" distribution)
- **Production data** = the data coming in RIGHT NOW from real users

We compare the two. If they're very different, the model may be making unreliable predictions.

**Analogy:** If you studied for an exam about European history, and the actual exam is about Asian history, you'll perform poorly — not because you're bad at history, but because the TEST (production) doesn't match what you PREPARED FOR (reference).

In [ ]:
# Cell 2 — Generate reference distribution (training data)
#
# This simulates what your training data looked like.
# Each distribution is chosen to match realistic patterns:
# - Age: Normal (bell curve centered at 35)
# - Income: Log-normal (most people earn middle, few earn very high)
# - Sessions: Poisson (count data, average 15 per month)
# - Satisfaction: Beta (bounded between 0-1, skewed toward high)

import numpy as np
import pandas as pd

np.random.seed(42)
N_REF = 5000
N_PROD = 5000

ref_data = pd.DataFrame({
    "feature_age": np.random.normal(35, 10, N_REF).clip(18, 80),
    "feature_income": np.random.lognormal(10.5, 0.6, N_REF),
    "feature_sessions": np.random.poisson(15, N_REF),
    "feature_satisfaction": np.random.beta(5, 2, N_REF),
    "prediction": np.random.binomial(1, 0.3, N_REF),
    "prediction_proba": np.random.beta(3, 7, N_REF),
})
ref_data["target"] = (ref_data["prediction_proba"] > 0.35).astype(int)

print(f"📊 Reference distribution: {ref_data.shape}")
print(ref_data.describe().round(2))

In [ ]:
# Cell 3 — Generate DRIFTED production distribution
#
# We deliberately make the production data DIFFERENT from the reference.
# This simulates real-world drift scenarios:
#   - Younger users (marketing campaign targeted Gen Z)
#   - Lower income (economy changed)
#   - More sessions (app went viral on social media)
#   - Lower satisfaction (new feature release had bugs)
#
# 🤔 Why simulate drift instead of using real data?
# So we can CONTROL exactly what drifts and by how much.
# This lets us validate that our detection methods actually work.

prod_data = pd.DataFrame({
    "feature_age": np.random.normal(28, 8, N_PROD).clip(18, 80),
    "feature_income": np.random.lognormal(10.0, 0.8, N_PROD),
    "feature_sessions": np.random.poisson(25, N_PROD),
    "feature_satisfaction": np.random.beta(3, 4, N_PROD),
    "prediction": np.random.binomial(1, 0.45, N_PROD),
    "prediction_proba": np.random.beta(4, 4, N_PROD),
})
prod_data["target"] = (prod_data["prediction_proba"] > 0.5).astype(int)

print(f"📊 Production distribution: {prod_data.shape}")
print(f"\n🔍 Drift signals injected:")
print(f"   Age mean: {ref_data['feature_age'].mean():.1f} → {prod_data['feature_age'].mean():.1f}")
print(f"   Income mean: {ref_data['feature_income'].mean():.0f} → {prod_data['feature_income'].mean():.0f}")
print(f"   Sessions mean: {ref_data['feature_sessions'].mean():.1f} → {prod_data['feature_sessions'].mean():.1f}")
print(f"   Satisfaction mean: {ref_data['feature_satisfaction'].mean():.2f} → {prod_data['feature_satisfaction'].mean():.2f}")

---

## 2 · Statistical Drift Detection

### 🤔 How Do We Mathematically Detect Drift?

We use statistical tests that answer: "Are these two distributions significantly different?"

| Test | How It Works | Best For | Output |
|------|-------------|----------|--------|
| **KS Test** | Measures max distance between two CDFs | Continuous features | p-value (< 0.05 = drift) |
| **PSI** | Measures divergence in binned distributions | All types | Score (> 0.2 = drift) |
| **Chi-squared** | Compares categorical distributions | Categorical features | p-value |
| **Jensen-Shannon** | Symmetric version of KL divergence | Probability distributions | Score (0-1) |

**🤔 What is a p-value?**  
It answers: "If there's no real drift, what's the probability of seeing data THIS different by pure chance?"  
- p < 0.05: "Very unlikely by chance → drift is real"
- p > 0.05: "Could easily be random fluctuation → no drift"

### ⚖️ Sensitivity Trade-off

| Too Sensitive (p < 0.001) | Too Lenient (p < 0.1) |
|--------------------------|----------------------|
| Alerts on tiny, meaningless changes | Misses moderate drift |
| Alert fatigue (team ignores alerts) | Model degrades silently |
| Best for: high-stakes systems | Best for: experimental/non-critical |

In [ ]:
# Cell 4 — EvidentlyAI automated drift report
#
# EvidentlyAI provides pre-built drift detection that:
# 1. Automatically selects the right statistical test per feature type
# 2. Generates visual HTML reports
# 3. Returns structured JSON results for programmatic use
#
# 🤔 Why use a library instead of writing our own tests?
# Because drift detection has many edge cases:
# - Different tests work better for different data types
# - Sample size affects test sensitivity
# - Visualization helps communicate findings to stakeholders

from evidently.report import Report
from evidently.metrics import DatasetDriftMetric, DataDriftTable

drift_report = Report(metrics=[DatasetDriftMetric(), DataDriftTable()])
drift_report.run(reference_data=ref_data.drop(columns=["target"]), current_data=prod_data.drop(columns=["target"]))
drift_report.save_html("drift_report.html")

results = drift_report.as_dict()
dataset_drift = results["metrics"][0]["result"]
print(f"📋 Dataset Drift Report:")
print(f"   Drift detected: {'🔴 YES' if dataset_drift['dataset_drift'] else '🟢 NO'}")
print(f"   Drifted features: {dataset_drift.get('number_of_drifted_columns', 'N/A')}")
print(f"   Share drifted: {dataset_drift.get('share_of_drifted_columns', 'N/A')}")
print(f"   Report saved: drift_report.html")

In [ ]:
# Cell 5 — Per-feature KS test analysis
#
# We manually compute KS tests to understand WHICH features drifted and by HOW MUCH.
# This is essential for ROOT CAUSE ANALYSIS:
#   "We know drift happened — but WHERE exactly?"

from scipy import stats

features = ["feature_age", "feature_income", "feature_sessions", "feature_satisfaction", "prediction_proba"]

print(f"{'Feature':<25} {'KS Stat':>10} {'KS p-val':>10} {'Drift?':>8}")
print("─" * 60)

drift_flags = {}
for feat in features:
    ks_stat, ks_pval = stats.ks_2samp(ref_data[feat], prod_data[feat])
    drifted = ks_pval < 0.05
    drift_flags[feat] = drifted
    indicator = "🔴" if drifted else "🟢"
    print(f"{feat:<25} {ks_stat:>10.4f} {ks_pval:>10.4e} {indicator:>8}")

print(f"\n📊 {sum(drift_flags.values())}/{len(drift_flags)} features show statistically significant drift")

In [ ]:
# Cell 6 — Visualize drift distributions side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feat in enumerate(features):
    ax = axes[idx]
    ax.hist(ref_data[feat], bins=50, alpha=0.5, label="Reference", color="#2196F3", density=True)
    ax.hist(prod_data[feat], bins=50, alpha=0.5, label="Production", color="#FF5722", density=True)
    ax.set_title(f"{feat}\n{'🔴 DRIFT' if drift_flags[feat] else '🟢 OK'}", fontsize=12, fontweight="bold")
    ax.legend()
    ax.set_ylabel("Density")

axes[-1].axis("off")
plt.suptitle("Reference vs Production Distributions", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("drift_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Drift visualization saved")

---

## 3 · Embedding-Space Drift Analysis

### 🤔 Why Monitor Embeddings Separately?

Traditional features (age, income) are easy to visualize and test. But LLMs and deep learning models use **embeddings** — 384-dimensional vectors that represent meaning.

**You can't just look at individual dimensions.** A shift in dimension 47 might mean nothing. But a shift in the OVERALL DISTRIBUTION of embedding vectors can indicate:
- New types of queries the model hasn't seen before
- Language distribution changes (more non-English requests)
- Domain shift (users asking about topics outside training distribution)

### Metrics for Embedding Drift

| Metric | What It Measures | Interpretation |
|--------|-----------------|----------------|
| **Centroid cosine similarity** | Shift in the "center" of the embedding cloud | < 0.95 → significant shift |
| **MMD** | Overall distribution difference (kernel method) | Higher = more different |
| **Intra-cluster similarity** | How "tight" each distribution is | Decreasing = more variety/uncertainty |

In [ ]:
# Cell 7 — Generate reference and production embeddings
from sklearn.metrics.pairwise import cosine_similarity

EMBED_DIM = 384
N_EMBED = 1000

ref_embeds = np.random.randn(N_EMBED, EMBED_DIM).astype(np.float32)
ref_embeds = ref_embeds / np.linalg.norm(ref_embeds, axis=1, keepdims=True)

drift_direction = np.random.randn(EMBED_DIM).astype(np.float32)
drift_direction = drift_direction / np.linalg.norm(drift_direction)
prod_embeds = np.random.randn(N_EMBED, EMBED_DIM).astype(np.float32)
prod_embeds += 0.5 * drift_direction
prod_embeds *= 1.3
prod_embeds = prod_embeds / np.linalg.norm(prod_embeds, axis=1, keepdims=True)

print(f"📐 Embedding dimensions: {EMBED_DIM}, Samples: {N_EMBED}")

In [ ]:
# Cell 8 — Compute embedding drift metrics

ref_centroid = ref_embeds.mean(axis=0)
prod_centroid = prod_embeds.mean(axis=0)
centroid_similarity = cosine_similarity([ref_centroid], [prod_centroid])[0][0]

ref_intra = cosine_similarity(ref_embeds[:200]).mean()
prod_intra = cosine_similarity(prod_embeds[:200]).mean()
cross_sim = cosine_similarity(ref_embeds[:200], prod_embeds[:200]).mean()

def compute_mmd(X, Y, gamma=1.0):
    from sklearn.metrics.pairwise import rbf_kernel
    return rbf_kernel(X, X, gamma=gamma).mean() + rbf_kernel(Y, Y, gamma=gamma).mean() - 2 * rbf_kernel(X, Y, gamma=gamma).mean()

mmd_score = compute_mmd(ref_embeds[:500], prod_embeds[:500], gamma=1.0 / EMBED_DIM)

print(f"📐 Embedding Drift Metrics:")
print(f"   Centroid cosine similarity:  {centroid_similarity:.4f}")
print(f"   Ref intra-similarity:  {ref_intra:.4f}")
print(f"   Prod intra-similarity: {prod_intra:.4f}")
print(f"   Cross-set similarity:  {cross_sim:.4f}")
print(f"   MMD score:             {mmd_score:.6f}")
print(f"\n   {'🔴 DRIFT DETECTED' if centroid_similarity < 0.95 else '🟢 NO DRIFT'} (threshold: 0.95)")

In [ ]:
# Cell 9 — t-SNE visualization of embedding drift
#
# 🤔 What is t-SNE?
# t-SNE reduces 384 dimensions to 2 dimensions for visualization.
# Points that are close in 384D space appear close in 2D.
#
# If reference (blue) and production (red) clusters overlap → no drift.
# If they're separated → drift detected.

from sklearn.manifold import TSNE

combined = np.vstack([ref_embeds[:500], prod_embeds[:500]])
labels = ["Reference"] * 500 + ["Production"] * 500

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
reduced = tsne.fit_transform(combined)

fig, ax = plt.subplots(figsize=(10, 8))
for label, color in [("Reference", "#2196F3"), ("Production", "#FF5722")]:
    mask = [l == label for l in labels]
    ax.scatter(reduced[mask, 0], reduced[mask, 1], c=color, alpha=0.5, s=15, label=label)

ax.set_title("Embedding Space: Reference vs Production (t-SNE)", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig("embedding_drift_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 4 · Automated Retraining Callback

### 🤔 What Happens When Drift Is Detected?

Three possible responses:

| Response | When | Action |
|----------|------|--------|
| **Alert only** | Minor drift, one feature | Slack/email notification, human reviews |
| **Auto-retrain** | Significant drift, multiple features | Pipeline automatically kicks off retraining with fresh data |
| **Rollback** | Critical drift, model unsafe | Revert to previous model version immediately |

### 🤔 What if We Auto-Retrain Too Often?

**Risk:** Costs pile up (GPU hours, engineering time) and each retraining is a risk (new model might be worse). Set clear thresholds: "only retrain if >50% of features drift AND centroid similarity drops below 0.95."

### 🤔 What if We Never Retrain?

**Risk:** Model performance degrades silently. Users get bad predictions. Revenue drops. By the time you notice, trust is lost.

In [ ]:
# Cell 10 — Alert threshold evaluation
import json

ALERT_CONFIG = {
    "drift_share_threshold": 0.5,
    "centroid_similarity_min": 0.95,
    "mmd_max": 0.01,
    "prediction_proba_ks_max": 0.1,
}

drift_share = sum(drift_flags.values()) / len(drift_flags)
ks_stat_proba = stats.ks_2samp(ref_data["prediction_proba"], prod_data["prediction_proba"]).statistic

alerts = []
if drift_share > ALERT_CONFIG["drift_share_threshold"]:
    alerts.append({"severity": "HIGH", "type": "feature_drift", "value": drift_share, "threshold": ALERT_CONFIG["drift_share_threshold"]})
if centroid_similarity < ALERT_CONFIG["centroid_similarity_min"]:
    alerts.append({"severity": "CRITICAL", "type": "embedding_drift", "value": centroid_similarity, "threshold": ALERT_CONFIG["centroid_similarity_min"]})
if mmd_score > ALERT_CONFIG["mmd_max"]:
    alerts.append({"severity": "HIGH", "type": "mmd_exceeded", "value": mmd_score, "threshold": ALERT_CONFIG["mmd_max"]})
if ks_stat_proba > ALERT_CONFIG["prediction_proba_ks_max"]:
    alerts.append({"severity": "HIGH", "type": "prediction_drift", "value": ks_stat_proba, "threshold": ALERT_CONFIG["prediction_proba_ks_max"]})

print(f"🚨 Alert Evaluation ({len(alerts)} triggered):")
for a in alerts:
    icon = "🔴" if a["severity"] == "CRITICAL" else "🟠"
    print(f"   {icon} [{a['severity']}] {a['type']}: {a['value']:.4f} (threshold: {a['threshold']})")

should_retrain = len(alerts) >= 2
print(f"\n   {'🔁 RETRAINING TRIGGERED' if should_retrain else '✅ Within tolerance'}")

In [ ]:
# Cell 11 — Simulate callback server
import threading
from http.server import HTTPServer, BaseHTTPRequestHandler

received_callbacks = []

class CallbackHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length)) if length else {}
        received_callbacks.append(body)
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b'{"status": "accepted"}')
    def log_message(self, format, *args): pass

server = HTTPServer(("127.0.0.1", 9999), CallbackHandler)
thread = threading.Thread(target=server.handle_request, daemon=True)
thread.start()
print("🖥️ Callback server started on :9999")

In [ ]:
# Cell 12 — Fire retraining webhook and verify
import requests, time
from datetime import datetime

if should_retrain:
    payload = {
        "event": "retraining_trigger",
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "reason": "Multiple drift thresholds exceeded",
        "alerts": alerts,
        "action": {"pipeline": "mlops-training-pipeline",
                   "parameters": {"raw_data_path": "s3://mlops-data/raw/latest/", "include_production_feedback": True}},
        "monitoring_snapshot": {"drift_share": drift_share, "centroid_similarity": float(centroid_similarity), "mmd_score": float(mmd_score)},
    }
    response = requests.post("http://127.0.0.1:9999/retrain", json=payload, timeout=5)
    print(f"🔁 Retraining webhook fired! Status: {response.status_code}")
    print(json.dumps(payload, indent=2, default=str))
else:
    print("✅ No retraining needed")

time.sleep(1)
print(f"\n📬 Callbacks received: {len(received_callbacks)}")
server.server_close()

---

## 5 -- A/B Testing & Experimentation for AI Models

### Why A/B Testing Is Different for AI

Traditional A/B testing (web buttons): measure click-through rate after 1 hour.  
AI A/B testing: measure model quality across THOUSANDS of diverse inputs over DAYS.

### Deployment Strategies

| Strategy | Risk | Speed | Best For |
|----------|:---:|:---:|----------|
| **Shadow mode** | Zero (no user impact) | Slow (need time to collect data) | First test of new model |
| **Canary release** (5% traffic) | Very low | Medium | Production validation |
| **Blue-green** (50/50 split) | Medium | Fast | Final comparison |
| **Full rollout** | High | Immediate | After all tests pass |

### Shadow Mode: Running Models Without User Impact

```
User Request ---> Model A (current, serves response)
              \-> Model B (new, runs silently, logs only)

Compare: latency, output quality, cost
Decision: If B is better AND cheaper -> promote to canary
```


In [ ]:
# Cell -- A/B test simulation
import numpy as np
np.random.seed(42)

n_queries = 500

# Model A (current production model)
model_a_latency = np.random.normal(450, 80, n_queries)   # ms
model_a_quality = np.random.normal(0.82, 0.05, n_queries)  # quality score
model_a_cost = 0.003  # per query

# Model B (new candidate)
model_b_latency = np.random.normal(320, 60, n_queries)   # ms (faster!)
model_b_quality = np.random.normal(0.85, 0.04, n_queries)  # (better!)
model_b_cost = 0.005  # per query (BUT more expensive)

# Statistical comparison
from scipy import stats
_, p_latency = stats.ttest_ind(model_a_latency, model_b_latency)
_, p_quality = stats.ttest_ind(model_a_quality, model_b_quality)

print('A/B Test Results (500 shadow queries):')
print(f'\n  Metric       | Model A (current) | Model B (new)   | Winner')
print(f'  -------------|-------------------|-----------------|-------')
print(f'  Latency (ms) | {model_a_latency.mean():.0f} +/- {model_a_latency.std():.0f}       | {model_b_latency.mean():.0f} +/- {model_b_latency.std():.0f}      | B (p={p_latency:.4f})')
print(f'  Quality      | {model_a_quality.mean():.3f} +/- {model_a_quality.std():.3f}   | {model_b_quality.mean():.3f} +/- {model_b_quality.std():.3f}  | B (p={p_quality:.4f})')
print(f'  Cost/query   | ${model_a_cost}           | ${model_b_cost}           | A (cheaper)')
print(f'\n  Decision: Model B is faster and higher quality.')
print(f'  BUT costs 67% more per query. At 100K queries/day:')
print(f'    Model A: ${model_a_cost * 100000:.0f}/day')
print(f'    Model B: ${model_b_cost * 100000:.0f}/day (+${(model_b_cost - model_a_cost) * 100000:.0f})')
print(f'  Proceed to canary release if quality gain justifies cost increase.')


---

## 6 -- Observability: OpenTelemetry & Structured Logging

### Why Standard Logging Is Not Enough for AI

Traditional log: `2026-03-01 INFO: Request processed`  
AI systems need: `2026-03-01 INFO: model=gpt-4o latency_ms=2340 tokens=1847 cost=$0.023 cache_hit=false trace_id=abc123`

### The Three Pillars of Observability

| Pillar | What It Captures | Tool |
|--------|-----------------|------|
| **Logs** | What happened (text events) | Structured JSON logs |
| **Metrics** | How much (counters, gauges) | Prometheus + Grafana |
| **Traces** | How long, which path | OpenTelemetry + Langfuse |

### Structured Logging for AI Services

```python
import structlog
logger = structlog.get_logger()

logger.info('inference_complete',
    model='gpt-4o',
    latency_ms=2340,
    prompt_tokens=512,
    completion_tokens=1335,
    cost_usd=0.023,
    cache_hit=False,
    trace_id='abc-123-def',
)
# Output: {"event": "inference_complete", "model": "gpt-4o", "latency_ms": 2340, ...}
```

### Key Metrics to Dashboard for AI Systems

| Metric | Alert If | Time Window |
|--------|---------|-------------|
| P50 latency | > 2s | 5 min |
| P99 latency | > 10s | 5 min |
| Error rate | > 5% | 1 min |
| Token cost | > daily budget | 1 hour |
| Cache hit rate | < 20% | 1 hour |
| Drift score | > threshold | 1 day |


---

## 🎯 Summary & Key Takeaways

| Step | Action | Tool | Why |
|------|--------|------|-----|
| 1 | Reference vs production comparison | NumPy + Pandas | Establish baseline for drift detection |
| 2 | Statistical drift analysis (KS, EvidentlyAI) | EvidentlyAI + SciPy | Quantify exactly what drifted |
| 3 | Embedding drift (cosine, MMD, t-SNE) | scikit-learn | Detect LLM-specific drift |
| 4 | Automated retraining webhook | HTTP server | Close the feedback loop automatically |

### 🧠 Key Questions

1. **"How big must the drift be to matter?"** → It depends on your model's sensitivity. A/B test to calibrate thresholds.
2. **"Should I retrain or fine-tune?"** → If drift is gradual, fine-tune on recent data. If drift is sudden, investigate the root cause first.
3. **"What if drift is intentional?"** → (e.g., new user segment) Update your reference data to include the new distribution.

**Next →** `37_llm_observability_cost.ipynb` — LLM Observability & Cost Management
